In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Small LSTM Network to Generate Text (PyTorch)
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# ── data ──────────────────────────────────────────────────────────────────────
filename = "/romeo_and_juliet.txt"
raw_text = open(filename, 'r', encoding='utf-8').read().lower()

chars = sorted(set(raw_text))
char_to_int = {c: i for i, c in enumerate(chars)}
n_chars = len(raw_text)
n_vocab = len(chars)
print(f"Total Characters: {n_chars}  |  Total Vocab: {n_vocab}")

seq_length = 100
dataX, dataY = [], []
for i in range(n_chars - seq_length):
    dataX.append([char_to_int[c] for c in raw_text[i:i + seq_length]])
    dataY.append(char_to_int[raw_text[i + seq_length]])
print(f"Total Patterns: {len(dataX)}")

X = np.array(dataX, dtype=np.float32).reshape(-1, seq_length, 1) / n_vocab
y = np.array(dataY, dtype=np.int64)

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader  = DataLoader(dataset, batch_size=128, shuffle=True)

# ── model ─────────────────────────────────────────────────────────────────────
class CharLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model     = CharLSTM(n_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# ── training ──────────────────────────────────────────────────────────────────
best_loss = float('inf')
epochs    = 1

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{epochs}", unit="batch")
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataset)
    bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        ckpt = f"weights-improvement-{epoch:02d}-{avg_loss:.4f}.pt"
        torch.save(model.state_dict(), ckpt)
        tqdm.write(f"  -> saved {ckpt}")

Total Characters: 25440  |  Total Vocab: 42
Total Patterns: 25340
Using device: cuda


Epoch 01/1: 100%|██████████| 198/198 [00:03<00:00, 61.78batch/s, loss=3.0476]

  -> saved weights-improvement-01-3.1200.pt


In [ ]:
# Load LSTM network and generate text (PyTorch)
import sys
import glob
import numpy as np
import torch
import torch.nn as nn

# ── data ──────────────────────────────────────────────────────────────────────
filename = "/romeo_and_juliet.txt"
raw_text = open(filename, 'r', encoding='utf-8').read().lower()

chars       = sorted(set(raw_text))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}
n_vocab     = len(chars)
n_chars     = len(raw_text)

seq_length = 100
dataX = []
for i in range(n_chars - seq_length):
    dataX.append([char_to_int[c] for c in raw_text[i:i + seq_length]])

# ── model ─────────────────────────────────────────────────────────────────────
class CharLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = CharLSTM(n_vocab).to(device)

checkpoints = glob.glob("weights-improvement-*.pt")
if not checkpoints:
    sys.exit("No checkpoint found.")
checkpoint = min(checkpoints, key=lambda f: float(f.rsplit('-', 1)[-1][:-3]))
print(f"Loading: {checkpoint}")
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()

# ── generate ──────────────────────────────────────────────────────────────────
start   = np.random.randint(0, len(dataX))
pattern = list(dataX[start])
print('Seed:')
print('"', ''.join(int_to_char[v] for v in pattern), '"')
print("\nGenerated text:")

for _ in range(500):
    x = torch.tensor(pattern, dtype=torch.float32).reshape(1, seq_length, 1) / n_vocab
    with torch.no_grad():
        logits = model(x.to(device))
    idx = int(torch.argmax(logits, dim=1).item())
    sys.stdout.write(int_to_char[idx])
    sys.stdout.flush()
    pattern.append(idx)
    pattern = pattern[1:]

print("\nDone.")

Loading: weights-improvement-01-3.1200.pt
Seed:
" low; i can read.

 [_he reads the letter._]

_signior martino and his wife and daughters;
county ans "

Generated text:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
Done.


In [ ]:
# Word-token LSTM to Generate Text (PyTorch)
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from nltk.tokenize import wordpunct_tokenize

# ── data ──────────────────────────────────────────────────────────────────────
raw_text       = open("/romeo_and_juliet.txt", 'r', encoding='utf-8').read().lower()
tokenized_text = wordpunct_tokenize(raw_text)
tokens         = sorted(dict.fromkeys(tokenized_text))

tok_to_int   = {t: i for i, t in enumerate(tokens)}
n_tokens     = len(tokenized_text)
n_token_vocab = len(tokens)
print(f"Total Tokens: {n_tokens}  |  Unique Tokens (Vocab): {n_token_vocab}")

seq_length = 100
dataX, dataY = [], []
for i in range(n_tokens - seq_length):
    dataX.append([tok_to_int[t] for t in tokenized_text[i:i + seq_length]])
    dataY.append(tok_to_int[tokenized_text[i + seq_length]])
print(f"Total Patterns: {len(dataX)}")

X = np.array(dataX, dtype=np.float32).reshape(-1, seq_length, 1) / n_token_vocab
y = np.array(dataY, dtype=np.int64)

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader  = DataLoader(dataset, batch_size=128, shuffle=True)

# ── model ─────────────────────────────────────────────────────────────────────
class TokenLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model     = TokenLSTM(n_token_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# ── training ──────────────────────────────────────────────────────────────────
best_loss = float('inf')
epochs    = 1

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{epochs}", unit="batch")
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataset)
    bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        ckpt = f"big-token-model-{epoch:02d}-{avg_loss:.4f}.pt"
        torch.save(model.state_dict(), ckpt)
        tqdm.write(f"  -> saved {ckpt}")

Total Tokens: 6117  |  Unique Tokens (Vocab): 1246
Total Patterns: 6017
Using device: cuda


Epoch 01/1: 100%|██████████| 48/48 [00:00<00:00, 68.90batch/s, loss=2.9374]


  -> saved big-token-model-01-6.1834.pt


In [ ]:
# Load word-token LSTM network and generate text (PyTorch)
import sys
import glob
import numpy as np
import torch
import torch.nn as nn
from nltk.tokenize import wordpunct_tokenize

# ── data ──────────────────────────────────────────────────────────────────────
raw_text       = open("/romeo_and_juliet.txt", 'r', encoding='utf-8').read().lower()
tokenized_text = wordpunct_tokenize(raw_text)
tokens         = sorted(dict.fromkeys(tokenized_text))

tok_to_int    = {t: i for i, t in enumerate(tokens)}
int_to_tok    = {i: t for i, t in enumerate(tokens)}
n_tokens      = len(tokenized_text)
n_token_vocab = len(tokens)

seq_length = 100
dataX = []
for i in range(n_tokens - seq_length):
    dataX.append([tok_to_int[t] for t in tokenized_text[i:i + seq_length]])

# ── model ─────────────────────────────────────────────────────────────────────
class TokenLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = TokenLSTM(n_token_vocab).to(device)

checkpoints = glob.glob("big-token-model-*.pt")
if not checkpoints:
    sys.exit("No checkpoint found.")
checkpoint = min(checkpoints, key=lambda f: float(f.rsplit('-', 1)[-1][:-3]))
print(f"Loading: {checkpoint}")
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()

# ── generate ──────────────────────────────────────────────────────────────────
start   = np.random.randint(0, len(dataX))
pattern = list(dataX[start])
print('Seed:')
print('"', ' '.join(int_to_tok[v] for v in pattern), '"')
print("\nGenerated text:")

for _ in range(100):
    x = torch.tensor(pattern, dtype=torch.float32).reshape(1, seq_length, 1) / n_token_vocab
    with torch.no_grad():
        logits = model(x.to(device))
    idx = int(torch.argmax(logits, dim=1).item())
    sys.stdout.write(int_to_tok[idx] + " ")
    sys.stdout.flush()
    pattern.append(idx)
    pattern = pattern[1:]

print("\nDone.")

Loading: big-token-model-01-6.1834.pt
Seed:
" ’ s hear our counsel . thou knowest my daughter ’ s of a pretty age . nurse . faith , i can tell her age unto an hour . lady capulet . she ’ s not fourteen . nurse . i ’ ll lay fourteen of my teeth , and yet , to my teen be it spoken , i have but four , she is not fourteen . how long is it now to lammas - tide ? lady capulet . a fortnight and odd days . nurse . even or odd , of all days in the "

Generated text:
, , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , 
Done.


In [ ]:
# Small LSTM Network to Generate Text (PyTorch)
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# ── data ──────────────────────────────────────────────────────────────────────
filename = "/romeo_and_juliet.txt"
raw_text = open(filename, 'r', encoding='utf-8').read().lower()

chars = sorted(set(raw_text))
char_to_int = {c: i for i, c in enumerate(chars)}
n_chars = len(raw_text)
n_vocab = len(chars)
print(f"Total Characters: {n_chars}  |  Total Vocab: {n_vocab}")

seq_length = 100
dataX, dataY = [], []
for i in range(n_chars - seq_length):
    dataX.append([char_to_int[c] for c in raw_text[i:i + seq_length]])
    dataY.append(char_to_int[raw_text[i + seq_length]])
print(f"Total Patterns: {len(dataX)}")

X = np.array(dataX, dtype=np.float32).reshape(-1, seq_length, 1) / n_vocab
y = np.array(dataY, dtype=np.int64)

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader  = DataLoader(dataset, batch_size=128, shuffle=True)

# ── model ─────────────────────────────────────────────────────────────────────
class CharLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model     = CharLSTM(n_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# ── training ──────────────────────────────────────────────────────────────────
best_loss = float('inf')
epochs    = 5

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{epochs}", unit="batch")
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataset)
    bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        ckpt = f"weights-improvement-{epoch:02d}-{avg_loss:.4f}.pt"
        torch.save(model.state_dict(), ckpt)
        tqdm.write(f"  -> saved {ckpt}")

Total Characters: 25440  |  Total Vocab: 42
Total Patterns: 25340
Using device: cuda


Epoch 01/5: 100%|██████████| 198/198 [00:02<00:00, 78.15batch/s, loss=3.1876]


  -> saved weights-improvement-01-3.1240.pt


Epoch 02/5: 100%|██████████| 198/198 [00:02<00:00, 82.15batch/s, loss=3.0739]


  -> saved weights-improvement-02-3.0785.pt


...

  -> saved weights-improvement-03-2.9958.pt


Epoch 04/5: 100%|██████████| 198/198 [00:02<00:00, 83.09batch/s, loss=2.8039]


  -> saved weights-improvement-04-2.8929.pt


Epoch 05/5: 100%|██████████| 198/198 [00:02<00:00, 82.85batch/s, loss=2.8729]

  -> saved weights-improvement-05-2.8433.pt


In [ ]:
# Word-token LSTM to Generate Text (PyTorch)
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from nltk.tokenize import wordpunct_tokenize

# ── data ──────────────────────────────────────────────────────────────────────
raw_text       = open("/romeo_and_juliet.txt", 'r', encoding='utf-8').read().lower()
tokenized_text = wordpunct_tokenize(raw_text)
tokens         = sorted(dict.fromkeys(tokenized_text))

tok_to_int   = {t: i for i, t in enumerate(tokens)}
n_tokens     = len(tokenized_text)
n_token_vocab = len(tokens)
print(f"Total Tokens: {n_tokens}  |  Unique Tokens (Vocab): {n_token_vocab}")

seq_length = 100
dataX, dataY = [], []
for i in range(n_tokens - seq_length):
    dataX.append([tok_to_int[t] for t in tokenized_text[i:i + seq_length]])
    dataY.append(tok_to_int[tokenized_text[i + seq_length]])
print(f"Total Patterns: {len(dataX)}")

X = np.array(dataX, dtype=np.float32).reshape(-1, seq_length, 1) / n_token_vocab
y = np.array(dataY, dtype=np.int64)

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader  = DataLoader(dataset, batch_size=128, shuffle=True)

# ── model ─────────────────────────────────────────────────────────────────────
class TokenLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model     = TokenLSTM(n_token_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# ── training ──────────────────────────────────────────────────────────────────
best_loss = float('inf')
epochs    = 5

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{epochs}", unit="batch")
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataset)
    bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        ckpt = f"big-token-model-{epoch:02d}-{avg_loss:.4f}.pt"
        torch.save(model.state_dict(), ckpt)
        tqdm.write(f"  -> saved {ckpt}")

Total Tokens: 6117  |  Unique Tokens (Vocab): 1246
Total Patterns: 6017
Using device: cuda


Epoch 01/5: 100%|██████████| 48/48 [00:00<00:00, 66.54batch/s, loss=5.0024]


  -> saved big-token-model-01-6.1889.pt


Epoch 02/5: 100%|██████████| 48/48 [00:00<00:00, 82.82batch/s, loss=5.2980]


  -> saved big-token-model-02-5.7466.pt


...

  -> saved big-token-model-03-5.7115.pt


Epoch 04/5: 100%|██████████| 48/48 [00:00<00:00, 83.56batch/s, loss=5.5389]


  -> saved big-token-model-04-5.7089.pt


Epoch 05/5: 100%|██████████| 48/48 [00:00<00:00, 83.62batch/s, loss=6.9092]

  -> saved big-token-model-05-5.7059.pt


In [ ]:
# Small LSTM Network to Generate Text (PyTorch)
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# ── data ──────────────────────────────────────────────────────────────────────
filename = "/romeo_and_juliet.txt"
raw_text = open(filename, 'r', encoding='utf-8').read().lower()

chars = sorted(set(raw_text))
char_to_int = {c: i for i, c in enumerate(chars)}
n_chars = len(raw_text)
n_vocab = len(chars)
print(f"Total Characters: {n_chars}  |  Total Vocab: {n_vocab}")

seq_length = 100
dataX, dataY = [], []
for i in range(n_chars - seq_length):
    dataX.append([char_to_int[c] for c in raw_text[i:i + seq_length]])
    dataY.append(char_to_int[raw_text[i + seq_length]])
print(f"Total Patterns: {len(dataX)}")

X = np.array(dataX, dtype=np.float32).reshape(-1, seq_length, 1) / n_vocab
y = np.array(dataY, dtype=np.int64)

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader  = DataLoader(dataset, batch_size=128, shuffle=True)

# ── model ─────────────────────────────────────────────────────────────────────
class CharLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model     = CharLSTM(n_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# ── training ──────────────────────────────────────────────────────────────────
best_loss = float('inf')
epochs    = 15

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{epochs}", unit="batch")
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataset)
    bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        ckpt = f"weights-improvement-{epoch:02d}-{avg_loss:.4f}.pt"
        torch.save(model.state_dict(), ckpt)
        tqdm.write(f"  -> saved {ckpt}")

Total Characters: 25440  |  Total Vocab: 42
Total Patterns: 25340
Using device: cuda


Epoch 01/15: 100%|██████████| 198/198 [00:02<00:00, 73.07batch/s, loss=3.0778]


  -> saved weights-improvement-01-3.1263.pt


Epoch 02/15: 100%|██████████| 198/198 [00:02<00:00, 83.48batch/s, loss=3.0450]


  -> saved weights-improvement-02-3.0796.pt


...

  -> saved weights-improvement-13-2.5785.pt


Epoch 14/15: 100%|██████████| 198/198 [00:02<00:00, 80.01batch/s, loss=2.5212]


  -> saved weights-improvement-14-2.5433.pt


Epoch 15/15: 100%|██████████| 198/198 [00:02<00:00, 80.36batch/s, loss=2.3941]

  -> saved weights-improvement-15-2.5108.pt


In [ ]:
# Word-token LSTM to Generate Text (PyTorch)
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from nltk.tokenize import wordpunct_tokenize

# ── data ──────────────────────────────────────────────────────────────────────
raw_text       = open("/romeo_and_juliet.txt", 'r', encoding='utf-8').read().lower()
tokenized_text = wordpunct_tokenize(raw_text)
tokens         = sorted(dict.fromkeys(tokenized_text))

tok_to_int   = {t: i for i, t in enumerate(tokens)}
n_tokens     = len(tokenized_text)
n_token_vocab = len(tokens)
print(f"Total Tokens: {n_tokens}  |  Unique Tokens (Vocab): {n_token_vocab}")

seq_length = 100
dataX, dataY = [], []
for i in range(n_tokens - seq_length):
    dataX.append([tok_to_int[t] for t in tokenized_text[i:i + seq_length]])
    dataY.append(tok_to_int[tokenized_text[i + seq_length]])
print(f"Total Patterns: {len(dataX)}")

X = np.array(dataX, dtype=np.float32).reshape(-1, seq_length, 1) / n_token_vocab
y = np.array(dataY, dtype=np.int64)

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader  = DataLoader(dataset, batch_size=128, shuffle=True)

# ── model ─────────────────────────────────────────────────────────────────────
class TokenLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model     = TokenLSTM(n_token_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# ── training ──────────────────────────────────────────────────────────────────
best_loss = float('inf')
epochs    = 15

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{epochs}", unit="batch")
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataset)
    bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        ckpt = f"big-token-model-{epoch:02d}-{avg_loss:.4f}.pt"
        torch.save(model.state_dict(), ckpt)
        tqdm.write(f"  -> saved {ckpt}")

Total Tokens: 6117  |  Unique Tokens (Vocab): 1246
Total Patterns: 6017
Using device: cuda


Epoch 01/15: 100%|██████████| 48/48 [00:00<00:00, 65.62batch/s, loss=2.6401]


  -> saved big-token-model-01-6.1848.pt


Epoch 02/15: 100%|██████████| 48/48 [00:00<00:00, 80.62batch/s, loss=4.5339]


  -> saved big-token-model-02-5.7420.pt


...

Epoch 10/15: 100%|██████████| 48/48 [00:00<00:00, 77.42batch/s, loss=6.7012]


  -> saved big-token-model-10-5.6905.pt


Epoch 12/15: 100%|██████████| 48/48 [00:00<00:00, 73.71batch/s, loss=7.1592]


  -> saved big-token-model-12-5.6871.pt


Epoch 15/15: 100%|██████████| 48/48 [00:00<00:00, 79.14batch/s, loss=6.2294]


In [ ]:
# Load LSTM network and generate text (PyTorch)
import sys
import glob
import numpy as np
import torch
import torch.nn as nn

# ── data (same preprocessing as training) ─────────────────────────────────────
filename = "/romeo_and_juliet.txt"
raw_text = open(filename, 'r', encoding='utf-8').read().lower()

chars       = sorted(set(raw_text))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}
n_vocab     = len(chars)
n_chars     = len(raw_text)

seq_length = 100
dataX = []
for i in range(n_chars - seq_length):
    dataX.append([char_to_int[c] for c in raw_text[i:i + seq_length]])

# ── model ─────────────────────────────────────────────────────────────────────
class CharLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = CharLSTM(n_vocab).to(device)

# pick the checkpoint with the lowest loss (last field before .pt)
checkpoints = glob.glob("weights-improvement-*.pt")
if not checkpoints:
    sys.exit("No checkpoint found. Train with lstm02_pt.py first.")
checkpoint = min(checkpoints, key=lambda f: float(f.rsplit('-', 1)[-1][:-3]))
print(f"Loading: {checkpoint}")
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()

# ── generate ──────────────────────────────────────────────────────────────────
start   = np.random.randint(0, len(dataX))
pattern = list(dataX[start])
print('Seed:')
print('"', ''.join(int_to_char[v] for v in pattern), '"')
print()

for _ in range(500):
    x = torch.tensor(pattern, dtype=torch.float32).reshape(1, seq_length, 1) / n_vocab
    with torch.no_grad():
        logits = model(x.to(device))
    idx = int(torch.argmax(logits, dim=1).item())
    sys.stdout.write(int_to_char[idx])
    sys.stdout.flush()
    pattern.append(idx)
    pattern = pattern[1:]

print("\nDone.")


Loading: weights-improvement-15-2.5108.pt
Seed:
" 

benvolio.
then she hath sworn that she will still live chaste?

romeo.
she hath, and in that spari "

 
and toe pore th the woe tor woet 

nempsnn.
to toe tor  and toe pare 
a dane toe carulet.

benvolio.
the  a tore th  aad bor barulet’  
nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
io  a do tor barulet.

nempson.
i
Done.


In [ ]:
# Load word-token LSTM network and generate text (PyTorch)
import sys
import glob
import numpy as np
import torch
import torch.nn as nn
from nltk.tokenize import wordpunct_tokenize

# ── data (same preprocessing as training) ─────────────────────────────────────
raw_text       = open("/romeo_and_juliet.txt", 'r', encoding='utf-8').read().lower()
tokenized_text = wordpunct_tokenize(raw_text)
tokens         = sorted(dict.fromkeys(tokenized_text))

tok_to_int    = {t: i for i, t in enumerate(tokens)}
int_to_tok    = {i: t for i, t in enumerate(tokens)}
n_tokens      = len(tokenized_text)
n_token_vocab = len(tokens)
print(f"Total Tokens: {n_tokens}  |  Unique Tokens (Vocab): {n_token_vocab}")

seq_length = 100
dataX = []
for i in range(n_tokens - seq_length):
    dataX.append([tok_to_int[t] for t in tokenized_text[i:i + seq_length]])

# ── model ─────────────────────────────────────────────────────────────────────
class TokenLSTM(nn.Module):
    def __init__(self, n_vocab, hidden=256, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(1, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, n_vocab)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = TokenLSTM(n_token_vocab).to(device)

# pick the checkpoint with the lowest loss
checkpoints = glob.glob("big-token-model-*.pt")
if not checkpoints:
    sys.exit("No checkpoint found. Train with lstm04_pt.py first.")
checkpoint = min(checkpoints, key=lambda f: float(f.rsplit('-', 1)[-1][:-3]))
print(f"Loading: {checkpoint}")
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()

# ── generate ──────────────────────────────────────────────────────────────────
start   = np.random.randint(0, len(dataX))
pattern = list(dataX[start])
print('Seed:')
print('"', ' '.join(int_to_tok[v] for v in pattern), '"')
print("\nGenerated text:")

for _ in range(100):
    x = torch.tensor(pattern, dtype=torch.float32).reshape(1, seq_length, 1) / n_token_vocab
    with torch.no_grad():
        logits = model(x.to(device))
    idx = int(torch.argmax(logits, dim=1).item())
    sys.stdout.write(int_to_tok[idx] + " ")
    sys.stdout.flush()
    pattern.append(idx)
    pattern = pattern[1:]

print("\nDone.")


Total Tokens: 6117  |  Unique Tokens (Vocab): 1246
Loading: big-token-model-14-5.6868.pt
Seed:
" scene iii . room in capulet ’ s house . enter lady capulet and nurse . lady capulet . nurse , where ’ s my daughter ? call her forth to me . nurse . now , by my maidenhead , at twelve year old , i bade her come . what , lamb ! what ladybird ! god forbid ! where ’ s this girl ? what , juliet ! enter juliet . juliet . how now , who calls ? nurse . your mother . juliet . madam , i am here . what is your will ? "

Generated text:
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Done.
